# Hard MoE Regressor (Straight-Through Gumbel-Softmax)

End-to-end hard Mixture-of-Experts (MoE) regressor with straight-through Gumbel-Softmax routing, two-phase grid search, and final comparison to the baseline.

## Imports + load baseline utilities

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple
import json
import hashlib
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

ARTIFACT_ROOT = NOTEBOOK_DIR / "artifacts" / "hard_moe"
CHECKPOINTS_DIR = ARTIFACT_ROOT / "checkpoints"
RESULTS_DIR = ARTIFACT_ROOT / "results"
PLOTS_DIR = ARTIFACT_ROOT / "plots"
HISTORIES_DIR = ARTIFACT_ROOT / "training_histories"
for _p in (CHECKPOINTS_DIR, RESULTS_DIR, PLOTS_DIR, HISTORIES_DIR):
    _p.mkdir(parents=True, exist_ok=True)

from data.constants import TARGET_COL
import data.encode_features as enc
from models.model import TabularRegressor
from models.train import TrainConfig, make_loaders


def _to_tuple(x):
    return tuple(x) if isinstance(x, list) else x


BASELINE_CONFIG_PATH = NOTEBOOK_DIR / "artifacts" / "baseline" / "results" / "config.json"
DEFAULT_BASELINE = {
    "encoder_hidden": (512, 256, 128, 64),
    "head_hidden": (),
    "dropout": 0.1,
    "lr": 3e-3,
    "weight_decay": 1e-4,
    "epochs": 200,
    "batch_size": 256,
    "seed": 0,
    "early_stop": {"patience": 25, "min_delta": 0.0, "restore_best": True},
    "scheduler": {"enabled": True, "mode": "min", "factor": 0.5, "patience": 8, "min_lr": 1e-6, "threshold": 1e-4},
}

BASELINE_CFG = DEFAULT_BASELINE.copy()
if BASELINE_CONFIG_PATH.exists():
    with BASELINE_CONFIG_PATH.open("r") as f:
        loaded = json.load(f)
    BASELINE_CFG.update(loaded)

BASELINE_CFG["encoder_hidden"] = _to_tuple(BASELINE_CFG.get("encoder_hidden", DEFAULT_BASELINE["encoder_hidden"]))
BASELINE_CFG["head_hidden"] = _to_tuple(BASELINE_CFG.get("head_hidden", DEFAULT_BASELINE["head_hidden"]))
BASELINE_CFG["dropout"] = float(BASELINE_CFG.get("dropout", DEFAULT_BASELINE["dropout"]))
BASELINE_CFG["lr"] = float(BASELINE_CFG.get("lr", DEFAULT_BASELINE["lr"]))
BASELINE_CFG["weight_decay"] = float(BASELINE_CFG.get("weight_decay", DEFAULT_BASELINE["weight_decay"]))
BASELINE_CFG["epochs"] = int(BASELINE_CFG.get("epochs", DEFAULT_BASELINE["epochs"]))
BASELINE_CFG["batch_size"] = int(BASELINE_CFG.get("batch_size", DEFAULT_BASELINE["batch_size"]))
BASELINE_CFG["seed"] = int(BASELINE_CFG.get("seed", DEFAULT_BASELINE["seed"]))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")
print(f"Baseline config path: {BASELINE_CONFIG_PATH}")
print(f"Baseline epochs: {BASELINE_CFG['epochs']} | batch_size: {BASELINE_CFG['batch_size']} | lr: {BASELINE_CFG['lr']}")

## Data loading and feature prep

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "all_features"
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

(
    X_train_np,
    X_val_np,
    X_test_np,
    y_train_np,
    y_val_np,
    y_test_np,
    feature_artifacts,
) = enc.prepare_features(train_df, val_df, test_df)

INPUT_DIM = X_train_np.shape[1]
BATCH_SIZE = int(BASELINE_CFG["batch_size"])

train_loader, val_loader = make_loaders(
    X_train_np,
    y_train_np,
    X_val_np,
    y_val_np,
    batch_size=BATCH_SIZE,
    separate=False,
)

test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test_np), torch.from_numpy(y_test_np)),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Input dim: {INPUT_DIM}")
print(f"Train/val/test: {X_train_np.shape}, {X_val_np.shape}, {X_test_np.shape}")

In [ ]:
CONFIG_SNAPSHOT_PATH = RESULTS_DIR / "config_snapshot.json"
CONFIG_SNAPSHOT_PATH.write_text(
    json.dumps(
        {
            "baseline_cfg": BASELINE_CFG,
            "input_dim": INPUT_DIM,
            "batch_size": BATCH_SIZE,
            "device": str(DEVICE),
        },
        indent=2,
        default=str,
    )
)
print(f"Saved config snapshot to: {CONFIG_SNAPSHOT_PATH}")

## HardMoERegressor

In [ ]:
class GateNet(nn.Module):
    def __init__(self, input_dim: int, num_experts: int, hidden_dims: Tuple[int, ...] = (128, 64), dropout: float = 0.1):
        super().__init__()
        layers = []
        dims = [input_dim, *hidden_dims]
        if hidden_dims:
            for d_in, d_out in zip(dims[:-1], dims[1:]):
                layers.append(nn.Linear(d_in, d_out))
                layers.append(nn.ReLU())
                if dropout and dropout > 0:
                    layers.append(nn.Dropout(dropout))
            layers.append(nn.Linear(dims[-1], num_experts))
        else:
            layers.append(nn.Linear(input_dim, num_experts))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class HardMoERegressor(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_experts: int,
        expert_hidden: Tuple[int, ...],
        dropout: float,
        gate_hidden: Tuple[int, ...] = (128, 64),
    ):
        super().__init__()
        self.num_experts = num_experts
        self.gate = GateNet(input_dim, num_experts, hidden_dims=gate_hidden, dropout=dropout)
        self.experts = nn.ModuleList(
            [
                TabularRegressor(
                    mode="shared",
                    input_dim=input_dim,
                    encoder_hidden=expert_hidden,
                    head_hidden=(),
                    dropout=dropout,
                    output_dim=1,
                )
                for _ in range(num_experts)
            ]
        )

    def forward(self, x: torch.Tensor, tau: float, use_gumbel: bool = True) -> Dict[str, torch.Tensor]:
        logits = self.gate(x)
        if use_gumbel and self.training:
            g_hard = F.gumbel_softmax(logits, tau=tau, hard=True)
        else:
            hard_idx = torch.argmax(logits, dim=1)
            g_hard = F.one_hot(hard_idx, num_classes=self.num_experts).float()
        p = torch.softmax(logits / tau, dim=1)

        y_all = torch.stack([expert(x) for expert in self.experts], dim=1)
        y_hat = (g_hard * y_all).sum(dim=1)

        return {
            "y_hat": y_hat,
            "y_all": y_all,
            "gate_logits": logits,
            "g_hard": g_hard,
            "p": p,
        }

## Training and evaluation helpers

In [ ]:
@dataclass
class EarlyStopConfig:
    patience: int = 25
    min_delta: float = 0.0
    restore_best: bool = True


@dataclass
class SchedConfig:
    enabled: bool = True
    mode: str = "min"
    factor: float = 0.5
    patience: int = 8
    min_lr: float = 1e-6
    threshold: float = 1e-4


class EarlyStopping:
    def __init__(self, cfg: EarlyStopConfig):
        self.cfg = cfg
        self.best = None
        self.bad = 0

    def step(self, value: float) -> bool:
        if self.best is None or value < (self.best - self.cfg.min_delta):
            self.best = value
            self.bad = 0
            return False
        self.bad += 1
        return self.bad >= self.cfg.patience


def set_deterministic_seed(seed: int) -> None:
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


def build_optimizer(model: nn.Module, lr: float, weight_decay: float) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


def build_scheduler(optimizer: torch.optim.Optimizer, cfg: SchedConfig | None):
    if cfg is None or not cfg.enabled:
        return None
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode=cfg.mode,
        factor=cfg.factor,
        patience=cfg.patience,
        min_lr=cfg.min_lr,
        threshold=cfg.threshold,
    )


def _usage_and_entropy(p: torch.Tensor, g_hard: torch.Tensor, eps: float = 1e-9):
    entropy = -(p * torch.log(p + eps)).sum(dim=1).mean()
    usage_soft = p.sum(dim=0)
    usage_hard = g_hard.sum(dim=0)
    return entropy, usage_soft, usage_hard


def run_moe_epoch(
    model: HardMoERegressor,
    loader,
    tau: float,
    entropy_weight: float,
    lambda_balance: float,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    loss_fn = nn.MSELoss()

    total_loss = 0.0
    total_mse = 0.0
    total_mae = 0.0
    total_entropy = 0.0
    total_balance = 0.0
    usage_soft_sum = None
    usage_hard_sum = None
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        out = model(xb, tau=tau, use_gumbel=is_train)
        y_hat = out["y_hat"]
        p = out["p"]
        g_hard = out["g_hard"]

        task_loss = loss_fn(y_hat, yb)
        entropy, usage_soft, usage_hard = _usage_and_entropy(p, g_hard)
        usage_soft_mean = usage_soft / float(yb.shape[0])
        balance = ((usage_soft_mean - (1.0 / model.num_experts)) ** 2).sum()

        loss = task_loss + (lambda_balance * balance) + (entropy_weight * entropy)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        batch_size = yb.shape[0]
        n_samples += batch_size
        total_loss += loss.item() * batch_size
        total_mse += torch.mean((y_hat - yb) ** 2).item() * batch_size
        total_mae += torch.mean(torch.abs(y_hat - yb)).item() * batch_size
        total_entropy += entropy.item() * batch_size
        total_balance += balance.item() * batch_size

        if usage_soft_sum is None:
            usage_soft_sum = usage_soft.detach().clone()
            usage_hard_sum = usage_hard.detach().clone()
        else:
            usage_soft_sum += usage_soft.detach()
            usage_hard_sum += usage_hard.detach()

    usage_soft_mean = (usage_soft_sum / max(n_samples, 1)).cpu().numpy()
    usage_hard_mean = (usage_hard_sum / max(n_samples, 1)).cpu().numpy()

    return {
        "loss": total_loss / max(n_samples, 1),
        "mse": total_mse / max(n_samples, 1),
        "mae": total_mae / max(n_samples, 1),
        "entropy": total_entropy / max(n_samples, 1),
        "balance": total_balance / max(n_samples, 1),
        "usage_soft": usage_soft_mean,
        "usage_hard": usage_hard_mean,
        "usage_soft_min": float(np.min(usage_soft_mean)),
        "usage_hard_min": float(np.min(usage_hard_mean)),
    }


def run_baseline_epoch(
    model: TabularRegressor,
    loader,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    loss_fn = nn.MSELoss()

    total_loss = 0.0
    total_mse = 0.0
    total_mae = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)
        loss = loss_fn(pred, yb)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        batch_size = yb.shape[0]
        n_samples += batch_size
        total_loss += loss.item() * batch_size
        total_mse += torch.mean((pred - yb) ** 2).item() * batch_size
        total_mae += torch.mean(torch.abs(pred - yb)).item() * batch_size

    return {
        "loss": total_loss / max(n_samples, 1),
        "mse": total_mse / max(n_samples, 1),
        "mae": total_mae / max(n_samples, 1),
    }


def train_moe(
    num_experts: int,
    tau: float,
    entropy_weight: float,
    lambda_balance: float,
    seed: int,
    max_epochs: int,
    early_stop_cfg: EarlyStopConfig | None = None,
    sched_cfg: SchedConfig | None = None,
):
    set_deterministic_seed(seed)

    model = HardMoERegressor(
        input_dim=INPUT_DIM,
        num_experts=num_experts,
        expert_hidden=BASELINE_CFG["encoder_hidden"],
        dropout=BASELINE_CFG["dropout"],
        gate_hidden=(128, 64),
    ).to(DEVICE)

    optimizer = build_optimizer(model, BASELINE_CFG["lr"], BASELINE_CFG["weight_decay"])
    scheduler = build_scheduler(optimizer, sched_cfg)
    stopper = EarlyStopping(early_stop_cfg) if early_stop_cfg else None

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_mse": [],
        "val_mse": [],
        "train_mae": [],
        "val_mae": [],
        "train_entropy": [],
        "val_entropy": [],
        "train_balance": [],
        "val_balance": [],
        "train_usage_soft": [],
        "train_usage_hard": [],
        "val_usage_soft": [],
        "val_usage_hard": [],
        "val_usage_soft_min": [],
        "val_usage_hard_min": [],
    }

    best_state = None
    best_val = None
    best_epoch = None

    for epoch in range(1, max_epochs + 1):
        train_metrics = run_moe_epoch(
            model,
            train_loader,
            tau=tau,
            entropy_weight=entropy_weight,
            lambda_balance=lambda_balance,
            device=DEVICE,
            optimizer=optimizer,
        )
        val_metrics = run_moe_epoch(
            model,
            val_loader,
            tau=tau,
            entropy_weight=entropy_weight,
            lambda_balance=lambda_balance,
            device=DEVICE,
            optimizer=None,
        )

        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["train_mse"].append(train_metrics["mse"])
        history["val_mse"].append(val_metrics["mse"])
        history["train_mae"].append(train_metrics["mae"])
        history["val_mae"].append(val_metrics["mae"])
        history["train_entropy"].append(train_metrics["entropy"])
        history["val_entropy"].append(val_metrics["entropy"])
        history["train_balance"].append(train_metrics["balance"])
        history["val_balance"].append(val_metrics["balance"])
        history["train_usage_soft"].append(train_metrics["usage_soft"])
        history["train_usage_hard"].append(train_metrics["usage_hard"])
        history["val_usage_soft"].append(val_metrics["usage_soft"])
        history["val_usage_hard"].append(val_metrics["usage_hard"])
        history["val_usage_soft_min"].append(val_metrics["usage_soft_min"])
        history["val_usage_hard_min"].append(val_metrics["usage_hard_min"])

        current_val = val_metrics["mse"]
        if best_val is None or current_val < best_val:
            best_val = current_val
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if scheduler is not None:
            scheduler.step(val_metrics["mse"])

        if stopper and stopper.step(val_metrics["mse"]):
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.best_epoch = best_epoch
    model.best_metric = best_val
    return model, history


def train_baseline(
    seed: int,
    max_epochs: int,
    early_stop_cfg: EarlyStopConfig | None = None,
    sched_cfg: SchedConfig | None = None,
):
    set_deterministic_seed(seed)

    model = TabularRegressor(
        mode="shared",
        input_dim=INPUT_DIM,
        encoder_hidden=BASELINE_CFG["encoder_hidden"],
        head_hidden=BASELINE_CFG["head_hidden"],
        dropout=BASELINE_CFG["dropout"],
        output_dim=1,
    ).to(DEVICE)

    optimizer = build_optimizer(model, BASELINE_CFG["lr"], BASELINE_CFG["weight_decay"])
    scheduler = build_scheduler(optimizer, sched_cfg)
    stopper = EarlyStopping(early_stop_cfg) if early_stop_cfg else None

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_mse": [],
        "val_mse": [],
        "train_mae": [],
        "val_mae": [],
    }

    best_state = None
    best_val = None
    best_epoch = None

    for epoch in range(1, max_epochs + 1):
        train_metrics = run_baseline_epoch(model, train_loader, device=DEVICE, optimizer=optimizer)
        val_metrics = run_baseline_epoch(model, val_loader, device=DEVICE, optimizer=None)

        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["train_mse"].append(train_metrics["mse"])
        history["val_mse"].append(val_metrics["mse"])
        history["train_mae"].append(train_metrics["mae"])
        history["val_mae"].append(val_metrics["mae"])

        current_val = val_metrics["mse"]
        if best_val is None or current_val < best_val:
            best_val = current_val
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if scheduler is not None:
            scheduler.step(val_metrics["mse"])

        if stopper and stopper.step(val_metrics["mse"]):
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.best_epoch = best_epoch
    model.best_metric = best_val
    return model, history


def evaluate_moe(model: HardMoERegressor, loader, tau: float):
    metrics = run_moe_epoch(
        model,
        loader,
        tau=tau,
        entropy_weight=0.0,
        lambda_balance=0.0,
        device=DEVICE,
        optimizer=None,
    )
    return metrics


def evaluate_baseline(model: TabularRegressor, loader):
    metrics = run_baseline_epoch(model, loader, device=DEVICE, optimizer=None)
    return metrics


def collect_moe_usage(model: HardMoERegressor, loader, tau: float):
    metrics = run_moe_epoch(
        model,
        loader,
        tau=tau,
        entropy_weight=0.0,
        lambda_balance=0.0,
        device=DEVICE,
        optimizer=None,
    )
    return metrics["usage_soft"], metrics["usage_hard"]

## Phase A grid search (coarse, fast)

In [ ]:
import itertools

NUM_EXPERTS_GRID = [2, 3, 4]
TAU_GRID = [0.5, 1.0, 2.0]
ENTROPY_WEIGHT_GRID = [0.0, 1e-3, 1e-2]
LAMBDA_BALANCE_GRID = [0.0, 1e-2, 1e-1]

PHASE_A_EPOCHS = max(20, int(BASELINE_CFG["epochs"] * 0.2))
PHASE_A_SEEDS = [BASELINE_CFG["seed"]]
PHASE_A_EARLY_STOP = EarlyStopConfig(patience=10, min_delta=0.0, restore_best=True)
PHASE_A_SCHED = SchedConfig(enabled=False)
TOP_K = 8

phase_a_results = []

for num_experts, tau, ent_w, lam_bal in itertools.product(
    NUM_EXPERTS_GRID, TAU_GRID, ENTROPY_WEIGHT_GRID, LAMBDA_BALANCE_GRID
):
    seed = PHASE_A_SEEDS[0]
    model, history = train_moe(
        num_experts=num_experts,
        tau=tau,
        entropy_weight=ent_w,
        lambda_balance=lam_bal,
        seed=seed,
        max_epochs=PHASE_A_EPOCHS,
        early_stop_cfg=PHASE_A_EARLY_STOP,
        sched_cfg=PHASE_A_SCHED,
    )

    best_idx = int(np.argmin(history["val_mse"]))
    best_val_mse = float(history["val_mse"][best_idx])
    best_val_mae = float(history["val_mae"][best_idx])
    mean_entropy = float(np.mean(history["val_entropy"]))
    min_usage_soft = float(np.min(history["val_usage_soft_min"]))
    min_usage_hard = float(np.min(history["val_usage_hard_min"]))

    phase_a_results.append(
        {
            "num_experts": num_experts,
            "tau": tau,
            "entropy_weight": ent_w,
            "lambda_balance": lam_bal,
            "seed": seed,
            "best_val_mse": best_val_mse,
            "best_val_mae": best_val_mae,
            "mean_val_entropy": mean_entropy,
            "min_usage_soft": min_usage_soft,
            "min_usage_hard": min_usage_hard,
        }
    )

phase_a_df = pd.DataFrame(phase_a_results)
phase_a_path = RESULTS_DIR / "phase_a_results.csv"
phase_a_df.to_csv(phase_a_path, index=False)

collapse_mask = (phase_a_df["min_usage_soft"] < 0.05) | (phase_a_df["min_usage_hard"] < 0.05)
phase_a_ranked = phase_a_df.loc[~collapse_mask].sort_values("best_val_mse")
phase_a_topk = phase_a_ranked.head(TOP_K).reset_index(drop=True)

print(f"Phase A results saved to: {phase_a_path}")
print("Top-K configs (Phase A):")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(phase_a_topk)

## Phase B grid search (re-evaluate Top-K)

In [ ]:
PHASE_B_EPOCHS = BASELINE_CFG["epochs"]
PHASE_B_SEEDS = [0, 1, 2]
PHASE_B_EARLY_STOP = EarlyStopConfig(
    patience=BASELINE_CFG.get("early_stop", {}).get("patience", 25),
    min_delta=BASELINE_CFG.get("early_stop", {}).get("min_delta", 0.0),
    restore_best=True,
)
PHASE_B_SCHED = SchedConfig(**BASELINE_CFG.get("scheduler", {})) if BASELINE_CFG.get("scheduler") else SchedConfig()

phase_b_rows = []
phase_b_histories = {}

for row in phase_a_topk.to_dict(orient="records"):
    cfg_key = f"E{row['num_experts']}_T{row['tau']}_EW{row['entropy_weight']}_LB{row['lambda_balance']}"
    seed_histories = []
    seed_rows = []

    for seed in PHASE_B_SEEDS:
        model, history = train_moe(
            num_experts=int(row["num_experts"]),
            tau=float(row["tau"]),
            entropy_weight=float(row["entropy_weight"]),
            lambda_balance=float(row["lambda_balance"]),
            seed=int(seed),
            max_epochs=PHASE_B_EPOCHS,
            early_stop_cfg=PHASE_B_EARLY_STOP,
            sched_cfg=PHASE_B_SCHED,
        )

        best_idx = int(np.argmin(history["val_mse"]))
        seed_rows.append(
            {
                "seed": seed,
                "best_val_mse": float(history["val_mse"][best_idx]),
                "best_val_mae": float(history["val_mae"][best_idx]),
                "mean_val_entropy": float(np.mean(history["val_entropy"])),
                "min_usage_soft": float(np.min(history["val_usage_soft_min"])),
                "min_usage_hard": float(np.min(history["val_usage_hard_min"])),
            }
        )
        seed_histories.append(history)

    phase_b_histories[cfg_key] = seed_histories
    seed_df = pd.DataFrame(seed_rows)
    phase_b_rows.append(
        {
            "config": cfg_key,
            "num_experts": row["num_experts"],
            "tau": row["tau"],
            "entropy_weight": row["entropy_weight"],
            "lambda_balance": row["lambda_balance"],
            "val_mse_mean": seed_df["best_val_mse"].mean(),
            "val_mse_std": seed_df["best_val_mse"].std(ddof=0),
            "val_mae_mean": seed_df["best_val_mae"].mean(),
            "val_mae_std": seed_df["best_val_mae"].std(ddof=0),
            "entropy_mean": seed_df["mean_val_entropy"].mean(),
            "entropy_std": seed_df["mean_val_entropy"].std(ddof=0),
            "min_usage_soft_mean": seed_df["min_usage_soft"].mean(),
            "min_usage_soft_std": seed_df["min_usage_soft"].std(ddof=0),
            "min_usage_hard_mean": seed_df["min_usage_hard"].mean(),
            "min_usage_hard_std": seed_df["min_usage_hard"].std(ddof=0),
        }
    )

phase_b_df = pd.DataFrame(phase_b_rows).sort_values(["val_mse_mean", "val_mse_std", "val_mae_mean"]).reset_index(drop=True)
phase_b_path = RESULTS_DIR / "phase_b_results.csv"
phase_b_df.to_csv(phase_b_path, index=False)

print(f"Phase B results saved to: {phase_b_path}")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(phase_b_df)

best_row = phase_b_df.iloc[0]
BEST_CONFIG = {
    "num_experts": int(best_row["num_experts"]),
    "tau": float(best_row["tau"]),
    "entropy_weight": float(best_row["entropy_weight"]),
    "lambda_balance": float(best_row["lambda_balance"]),
}

print("BEST_CONFIG:")
print(BEST_CONFIG)

# Plot validation MSE curves for the top few configs
TOP_PLOT = min(3, len(phase_b_df))
for idx in range(TOP_PLOT):
    row = phase_b_df.iloc[idx]
    cfg_key = row["config"]
    histories = phase_b_histories[cfg_key]
    min_len = min(len(h["val_mse"]) for h in histories)
    val_mse_stack = np.stack([np.array(h["val_mse"][:min_len]) for h in histories], axis=0)
    mean_curve = val_mse_stack.mean(axis=0)
    std_curve = val_mse_stack.std(axis=0)

    epochs = np.arange(1, min_len + 1)
    plt.figure(figsize=(6, 4))
    plt.plot(epochs, mean_curve, label="val MSE")
    plt.fill_between(epochs, mean_curve - std_curve, mean_curve + std_curve, alpha=0.2)
    plt.title(f"Phase B Val MSE: {cfg_key}")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.grid(True, alpha=0.3)
    plot_path = PLOTS_DIR / f"phase_b_val_mse_{idx+1}.png"
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"Saved: {plot_path}")

## Final training + baseline comparison

In [ ]:
FINAL_SEEDS = PHASE_B_SEEDS
FINAL_EPOCHS = BASELINE_CFG["epochs"]
FINAL_EARLY_STOP = EarlyStopConfig(
    patience=BASELINE_CFG.get("early_stop", {}).get("patience", 25),
    min_delta=BASELINE_CFG.get("early_stop", {}).get("min_delta", 0.0),
    restore_best=True,
)
FINAL_SCHED = PHASE_B_SCHED

moe_histories = []
moe_val_metrics = []
moe_test_metrics = []
moe_usage_soft = []
moe_usage_hard = []

for seed in FINAL_SEEDS:
    model, history = train_moe(
        num_experts=BEST_CONFIG["num_experts"],
        tau=BEST_CONFIG["tau"],
        entropy_weight=BEST_CONFIG["entropy_weight"],
        lambda_balance=BEST_CONFIG["lambda_balance"],
        seed=seed,
        max_epochs=FINAL_EPOCHS,
        early_stop_cfg=FINAL_EARLY_STOP,
        sched_cfg=FINAL_SCHED,
    )
    moe_histories.append(history)

    best_idx = int(np.argmin(history["val_mse"]))
    moe_val_metrics.append(
        {
            "val_mse": float(history["val_mse"][best_idx]),
            "val_mae": float(history["val_mae"][best_idx]),
            "entropy": float(history["val_entropy"][best_idx]),
            "min_usage_soft": float(history["val_usage_soft_min"][best_idx]),
            "min_usage_hard": float(history["val_usage_hard_min"][best_idx]),
        }
    )

    test_metrics = evaluate_moe(model, test_loader, tau=BEST_CONFIG["tau"])
    moe_test_metrics.append({"test_mse": test_metrics["mse"], "test_mae": test_metrics["mae"]})

    usage_soft, usage_hard = collect_moe_usage(model, val_loader, tau=BEST_CONFIG["tau"])
    moe_usage_soft.append(usage_soft)
    moe_usage_hard.append(usage_hard)

    ckpt_path = CHECKPOINTS_DIR / f"hard_moe_seed{seed}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": {**BEST_CONFIG, "epochs": FINAL_EPOCHS},
            "seed": seed,
            "input_dim": INPUT_DIM,
        },
        ckpt_path,
    )

baseline_histories = []
baseline_val_metrics = []
baseline_test_metrics = []

for seed in FINAL_SEEDS:
    model, history = train_baseline(
        seed=seed,
        max_epochs=FINAL_EPOCHS,
        early_stop_cfg=FINAL_EARLY_STOP,
        sched_cfg=FINAL_SCHED,
    )
    baseline_histories.append(history)

    best_idx = int(np.argmin(history["val_mse"]))
    baseline_val_metrics.append(
        {
            "val_mse": float(history["val_mse"][best_idx]),
            "val_mae": float(history["val_mae"][best_idx]),
        }
    )

    test_metrics = evaluate_baseline(model, test_loader)
    baseline_test_metrics.append({"test_mse": test_metrics["mse"], "test_mae": test_metrics["mae"]})

    ckpt_path = CHECKPOINTS_DIR / f"baseline_seed{seed}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": BASELINE_CFG,
            "seed": seed,
            "input_dim": INPUT_DIM,
        },
        ckpt_path,
    )

# Helper for mean +/- std formatting

def _mean_std(series: List[float]):
    arr = np.array(series, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))


def _fmt(mean: float, std: float):
    return f"{mean:.4f} +/- {std:.4f}"


# Validation curves: MSE and MAE

def _mean_std_curve(histories, key: str, max_len: int):
    stack = np.stack([np.array(h[key][:max_len]) for h in histories], axis=0)
    return stack.mean(axis=0), stack.std(axis=0)


def _plot_compare(histories_a, histories_b, key: str, label_a: str, label_b: str, ylabel: str, out_path: Path):
    min_len = min(min(len(h[key]) for h in histories_a), min(len(h[key]) for h in histories_b))
    mean_a, std_a = _mean_std_curve(histories_a, key, min_len)
    mean_b, std_b = _mean_std_curve(histories_b, key, min_len)
    epochs = np.arange(1, min_len + 1)

    plt.figure(figsize=(6, 4))
    plt.plot(epochs, mean_a, label=label_a)
    plt.fill_between(epochs, mean_a - std_a, mean_a + std_a, alpha=0.2)
    plt.plot(epochs, mean_b, label=label_b)
    plt.fill_between(epochs, mean_b - std_b, mean_b + std_b, alpha=0.2)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()


val_mse_plot = PLOTS_DIR / "final_val_mse_baseline_vs_moe.png"
val_mae_plot = PLOTS_DIR / "final_val_mae_baseline_vs_moe.png"

_plot_compare(baseline_histories, moe_histories, "val_mse", "baseline", "hard_moe", "Val MSE", val_mse_plot)
_plot_compare(baseline_histories, moe_histories, "val_mae", "baseline", "hard_moe", "Val MAE", val_mae_plot)

print(f"Saved: {val_mse_plot}")
print(f"Saved: {val_mae_plot}")

# Test performance comparison
baseline_test_df = pd.DataFrame(baseline_test_metrics)
moe_test_df = pd.DataFrame(moe_test_metrics)

baseline_test_mean, baseline_test_std = _mean_std(baseline_test_df["test_mse"])
moe_test_mean, moe_test_std = _mean_std(moe_test_df["test_mse"])

plt.figure(figsize=(5, 4))
plt.bar(["baseline", "moe"], [baseline_test_mean, moe_test_mean], yerr=[baseline_test_std, moe_test_std])
plt.ylabel("Test MSE")
plt.title("Test MSE (mean +/- std)")
plt.tight_layout()
bar_path = PLOTS_DIR / "final_test_mse_bar.png"
plt.savefig(bar_path, dpi=150)
plt.show()

baseline_test_mean, baseline_test_std = _mean_std(baseline_test_df["test_mae"])
moe_test_mean, moe_test_std = _mean_std(moe_test_df["test_mae"])

plt.figure(figsize=(5, 4))
plt.bar(["baseline", "moe"], [baseline_test_mean, moe_test_mean], yerr=[baseline_test_std, moe_test_std])
plt.ylabel("Test MAE")
plt.title("Test MAE (mean +/- std)")
plt.tight_layout()
bar_path_mae = PLOTS_DIR / "final_test_mae_bar.png"
plt.savefig(bar_path_mae, dpi=150)
plt.show()

print(f"Saved: {bar_path}")
print(f"Saved: {bar_path_mae}")

# Expert usage bars (mean across seeds)
moe_usage_soft_mean = np.mean(np.stack(moe_usage_soft, axis=0), axis=0)
moe_usage_hard_mean = np.mean(np.stack(moe_usage_hard, axis=0), axis=0)

plt.figure(figsize=(6, 4))
plt.bar(np.arange(len(moe_usage_hard_mean)), moe_usage_hard_mean)
plt.xlabel("Expert")
plt.ylabel("Hard routing fraction")
plt.title("Hard routing fractions (val)")
plt.tight_layout()
usage_hard_path = PLOTS_DIR / "moe_usage_hard.png"
plt.savefig(usage_hard_path, dpi=150)
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(np.arange(len(moe_usage_soft_mean)), moe_usage_soft_mean)
plt.xlabel("Expert")
plt.ylabel("Soft usage")
plt.title("Soft usage (val)")
plt.tight_layout()
usage_soft_path = PLOTS_DIR / "moe_usage_soft.png"
plt.savefig(usage_soft_path, dpi=150)
plt.show()

print(f"Saved: {usage_hard_path}")
print(f"Saved: {usage_soft_path}")

# Gate entropy vs epochs (MoE)
min_len = min(len(h["val_entropy"]) for h in moe_histories)
entropy_stack = np.stack([np.array(h["val_entropy"][:min_len]) for h in moe_histories], axis=0)
entropy_mean = entropy_stack.mean(axis=0)
entropy_std = entropy_stack.std(axis=0)

plt.figure(figsize=(6, 4))
plt.plot(np.arange(1, min_len + 1), entropy_mean)
plt.fill_between(
    np.arange(1, min_len + 1),
    entropy_mean - entropy_std,
    entropy_mean + entropy_std,
    alpha=0.2,
)
plt.xlabel("Epoch")
plt.ylabel("Gate entropy")
plt.title("MoE gate entropy vs epoch")
plt.tight_layout()
entropy_path = PLOTS_DIR / "moe_gate_entropy.png"
plt.savefig(entropy_path, dpi=150)
plt.show()

print(f"Saved: {entropy_path}")

# Summary table
baseline_val_df = pd.DataFrame(baseline_val_metrics)
moe_val_df = pd.DataFrame(moe_val_metrics)

summary_rows = []

baseline_val_mse_mean, baseline_val_mse_std = _mean_std(baseline_val_df["val_mse"])
baseline_val_mae_mean, baseline_val_mae_std = _mean_std(baseline_val_df["val_mae"])
baseline_test_mse_mean, baseline_test_mse_std = _mean_std(baseline_test_df["test_mse"])
baseline_test_mae_mean, baseline_test_mae_std = _mean_std(baseline_test_df["test_mae"])

summary_rows.append(
    {
        "model": "baseline",
        "val_mse": _fmt(baseline_val_mse_mean, baseline_val_mse_std),
        "val_mae": _fmt(baseline_val_mae_mean, baseline_val_mae_std),
        "test_mse": _fmt(baseline_test_mse_mean, baseline_test_mse_std),
        "test_mae": _fmt(baseline_test_mae_mean, baseline_test_mae_std),
        "mean_entropy": "-",
        "min_usage_soft": "-",
        "min_usage_hard": "-",
    }
)

moe_val_mse_mean, moe_val_mse_std = _mean_std(moe_val_df["val_mse"])
moe_val_mae_mean, moe_val_mae_std = _mean_std(moe_val_df["val_mae"])
moe_test_mse_mean, moe_test_mse_std = _mean_std(moe_test_df["test_mse"])
moe_test_mae_mean, moe_test_mae_std = _mean_std(moe_test_df["test_mae"])
moe_entropy_mean, moe_entropy_std = _mean_std(moe_val_df["entropy"])
moe_min_usage_soft_mean, moe_min_usage_soft_std = _mean_std(moe_val_df["min_usage_soft"])
moe_min_usage_hard_mean, moe_min_usage_hard_std = _mean_std(moe_val_df["min_usage_hard"])

summary_rows.append(
    {
        "model": "hard_moe",
        "val_mse": _fmt(moe_val_mse_mean, moe_val_mse_std),
        "val_mae": _fmt(moe_val_mae_mean, moe_val_mae_std),
        "test_mse": _fmt(moe_test_mse_mean, moe_test_mse_std),
        "test_mae": _fmt(moe_test_mae_mean, moe_test_mae_std),
        "mean_entropy": _fmt(moe_entropy_mean, moe_entropy_std),
        "min_usage_soft": _fmt(moe_min_usage_soft_mean, moe_min_usage_soft_std),
        "min_usage_hard": _fmt(moe_min_usage_hard_mean, moe_min_usage_hard_std),
    }
)

summary_df = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / "final_summary.csv"
summary_df.to_csv(summary_path, index=False)

with pd.option_context("display.max_columns", None):
    display(summary_df)

print(f"Saved summary to: {summary_path}")

## Plots + summary tables

In [ ]:
from IPython.display import Image, display

summary_path = RESULTS_DIR / "final_summary.csv"
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    display(summary_df)

plot_paths = [
    PLOTS_DIR / "final_val_mse_baseline_vs_moe.png",
    PLOTS_DIR / "final_val_mae_baseline_vs_moe.png",
    PLOTS_DIR / "final_test_mse_bar.png",
    PLOTS_DIR / "final_test_mae_bar.png",
    PLOTS_DIR / "moe_usage_hard.png",
    PLOTS_DIR / "moe_usage_soft.png",
    PLOTS_DIR / "moe_gate_entropy.png",
]

for path in plot_paths:
    if path.exists():
        display(Image(filename=str(path)))